In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_expenses_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #      "ExpensesId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "ExpensesKey string,"
        #      "PartyKey string,"
        #      "CDI string,"
        #      "ExposureKey string,"
        #      "AgreementKey string,"
        #      "LegalActionKey string,"
        #      "InsuranceKey string,"
        #      "ExpensesTypeCode string,"
        #      "ExpensesType string,"
        #      "ExpensesDate timestamp,"
        #      "TotalAmount decimal(20,6),"
        #      "NetAmount decimal(20,6),"
        #      "TaxAmount decimal(19,4),"
        #      "VatAmount decimal(20,6),"
        #      "InternalCost decimal(19,10),"
        #      "OtherExpenses decimal(19,10),"
        #      "ExpensesCategoryCode string,"
        #      "ExpensesCategory string,"
        #      "ExpensesStatusCode string,"
        #      "ExpensesStatus string,"
        #      "VATPercentage string,"
        #      "PacketCode int,"
        #      "Packet string,"
        #      "Comments string,"
        #      "LawyerKey int,"
        #      "ExternalCompanyPartyKey int,"
        #      "InvoiceDate timestamp,"
        #      "InvoiceNumber string,"
        #      "InvoiceApprovalStatusCode int,"
        #      "InvoiceApprovalStatus int,"
        #      "ExpensesValidationStatusCode int,"
        #      "ExpensesValidationStatus int,"
        #      "ExpensesApprovalStatusCode int,"
        #      "ExpensesApprovalStatus int,"
        #      "BusinessDate date"
        #     )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Expenses"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ============================== arxlegexp ====================
        arxlegexp_extra_filters = None
        if arxlegexp_extra_filters is not None:
            arxlegexp_extra_filters.append(date_filter)
            arxlegexp_final_filters = " AND ".join(arxlegexp_extra_filters)
        else:
            arxlegexp_final_filters = date_filter
        arxlegexp_selected_fields = [
            "ExpID",
            "PorfCode",
            "CustCDI",
            "ContNum",
            "ExpType",
            "PmtAmt",
            "ExpDate",
            "LegActID",
            "ExpNetAmt",
            "VATAmt",
            "ExpCatID",
            "VATPs",
            "ExpComments",
            "InvNumber",
            "ExpStatusID",
            "Businessdate",
        ]

        arxlegexp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxlegexp",
            arxlegexp_final_filters,
            arxlegexp_selected_fields,
        )

        # ============================== arxmdata ====================
        arxmdata_extra_filters = None
        if arxmdata_extra_filters is not None:
            arxmdata_extra_filters.append(date_filter)
            arxmdata_final_filters = " AND ".join(arxmdata_extra_filters)
        else:
            arxmdata_final_filters = date_filter
        arxmdata_selected_fields = [
            "LstItemRefCode",
            "LstItemDescr",
        ]

        arxmdata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxmdata",
            arxmdata_final_filters,
            arxmdata_selected_fields,
        )

        arxmdata_Porf = arxmdata.withColumnRenamed(
            "LstItemRefCode", "LstItemRefCode_Porf"
        ).withColumnRenamed("LstItemDescr", "LstItemDescr_Porf")

        arxmdata_ExpCat = arxmdata.withColumnRenamed(
            "LstItemRefCode", "LstItemRefCode_ExpCat"
        ).withColumnRenamed("LstItemDescr", "LstItemDescr_ExpCat")

        arxmdata_ExpStatus = arxmdata.withColumnRenamed(
            "LstItemRefCode", "LstItemRefCode_ExpStatus"
        ).withColumnRenamed("LstItemDescr", "LstItemDescr_ExpStatus")


        # ====================== Join Conditions =========================

        arxlegexp_arxmdata_Porf_join_condition = (
            arxlegexp["PorfCode"] == arxmdata_Porf["LstItemRefCode_Porf"]
        )

        arxlegexp_arxmdata_ExpCat_join_condition = (
            arxlegexp["ExpCatID"] == arxmdata_ExpCat["LstItemDescr_ExpCat"]
        )

        arxlegexp_arxmdata_ExpStatus_join_condition = (
            arxlegexp["ExpStatusID"] == arxmdata_ExpStatus["LstItemDescr_ExpStatus"]
        )

        arxlegexp_portfolio_join_condition = (
            substring(arxlegexp["CUSTCDI"], 1, 3) == portfolio["prefix"]
        )

        # ==================== Apply the Joins =========================

        Joined_df = (
            arxlegexp.join(
                arxmdata_Porf,
                arxlegexp_arxmdata_Porf_join_condition,
                "left",
            )
            .join(
                arxmdata_ExpCat,
                arxlegexp_arxmdata_ExpCat_join_condition,
                "left",
            )
            .join(
                arxmdata_ExpStatus,
                arxlegexp_arxmdata_ExpStatus_join_condition,
                "left",
            )
            .join(portfolio, arxlegexp_portfolio_join_condition, "left")
        )

        # ================= Set the Keys for Hashkey================

        ExpenseId__primary_key_fields = [
            arxlegexp.ExpID,
            lit("arotron"),
        ]

        # =============== Creation of Hashkeys ===================

        target_table_arotron = Joined_df.withColumn(
            "ExpensesId",
            sha2(
                concat_ws(";", *ExpenseId__primary_key_fields),
                256,
            ),
        ).withColumn(
            "AgreementKey",
            when(arxlegexp.ContNum.isNull(), lit(None)).otherwise(
                arxlegexp.ContNum.cast("string")
            ),
        )

        # =========== Selection of Final Columns ================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.ExpensesId.cast("string").alias("ExpensesId"),
            lit("arotron").cast("string").alias("SourceSystem"),
            coalesce(portfolio["unit_code_profits"], lit("-9998"))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio["PortfolioName"], lit("Core Wholesale"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            arxlegexp.ExpID.cast("string").alias("ExpensesKey"),
            arxlegexp.CustCDI.cast("string").alias("PartyKey"),
            arxlegexp.CustCDI.cast("string").alias("CDI"),
            lit(None).cast("string").alias("ExposureKey"),
            target_table_arotron.AgreementKey.cast("string").alias("AgreementKey"),
            arxlegexp.LegActID.cast("string").alias("LegalActionKey"),
            lit(None).cast("string").alias("InsuranceKey"),
            arxlegexp.ExpType.cast("string").alias("ExpensesTypeCode"),
            arxlegexp.ExpType.cast("string").alias("ExpensesType"),
            arxlegexp.ExpDate.cast("timestamp").alias("ExpensesDate"),
            arxlegexp.PmtAmt.cast("decimal(20,6)").alias("TotalAmount"),
            arxlegexp.ExpNetAmt.cast("decimal(20,6)").alias("NetAmount"),
            lit(None).cast("decimal(19,4)").alias("TaxAmount"),
            arxlegexp.VATAmt.cast("decimal(20,6)").alias("VatAmount"),
            lit(None).cast("decimal(19,10)").alias("InternalCost"),
            lit(None).alias("OtherExpenses"),
            arxmdata_ExpCat.LstItemRefCode_ExpCat.cast("string").alias(
                "ExpensesCategoryCode"
            ),
            arxmdata_ExpCat.LstItemDescr_ExpCat.cast("string").alias(
                "ExpensesCategory"
            ),
            arxmdata_ExpStatus.LstItemRefCode_ExpStatus.cast("string").alias(
                "ExpensesStatusCode"
            ),
            arxmdata_ExpStatus.LstItemDescr_ExpStatus.cast("string").alias(
                "ExpensesStatus"
            ),
            arxlegexp.VATPs.cast("string").alias("VATPercentage"),
            lit(None).cast("int").alias("PacketCode"),
            lit(None).cast("string").alias("Packet"),
            arxlegexp.ExpComments.cast("string").alias("Comments"),
            lit(None).cast("int").alias("LawyerKey"),
            lit(None).cast("int").alias("ExternalCompanyPartyKey"),
            lit(None).cast("timestamp").alias("InvoiceDate"),
            arxlegexp.InvNumber.cast("string").alias("InvoiceNumber"),
            lit(None).cast("int").alias("InvoiceApprovalStatusCode"),
            lit(None).cast("int").alias("InvoiceApprovalStatus"),
            lit(None).cast("int").alias("ExpensesValidationStatusCode"),
            lit(None).cast("int").alias("ExpensesValidationStatus"),
            lit(None).cast("int").alias("ExpensesApprovalStatusCode"),
            lit(None).cast("int").alias("ExpensesApprovalStatus"),
            arxlegexp.Businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_expenses_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)